In [1]:
import pandas as pd
import numpy as np
from dateutil import parser
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt

In [2]:
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

In [3]:
#Loading the csv file
df = pd.read_csv("data/jiji_housing_raw.csv")

df.head()

,title,property_size,bedrooms,bathrooms,furnishing,region,region_name,region_parent_name,is_boost,price
0,"3bdrm Penthouse in Brownstone Estate, Ikate fo...",150,3,3,semi-furnished,"Lekki, Ikate",Ikate,Lekki,False,"₦ 250,000,000"
1,"4bdrm Duplex in Redemption Estate, Owerri for ...",400,4,5,unfurnished,"Imo State, Owerri",Owerri,Imo State,premium,"₦ 200,000,000"
2,6bdrm Bungalow in Ikorodu Garage for sale,5000,6,6,unfurnished,"Ikorodu, Ikorodu Garage",Ikorodu Garage,Ikorodu,False,"₦ 65,000,000"
3,4bdrm Duplex in Opebi for sale,225,4,4,semi-furnished,"Ikeja, Opebi",Opebi,Ikeja,enterprise,"₦ 450,000,000"
4,"3bdrm House in Treasure Hilltop, Akesan for sale",300,3,4,unfurnished,"Alimosho, Akesan",Akesan,Alimosho,False,"₦ 85,000,000"


In [4]:
#Missing value check

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   title               1980 non-null   str  
 1   property_size       1980 non-null   int64
 2   bedrooms            1980 non-null   int64
 3   bathrooms           1980 non-null   int64
 4   furnishing          1980 non-null   str  
 5   region              1980 non-null   str  
 6   region_name         1980 non-null   str  
 7   region_parent_name  1978 non-null   str  
 8   is_boost            1980 non-null   str  
 9   price               1980 non-null   str  
dtypes: int64(3), str(7)
memory usage: 373.9 KB


In [5]:
df.columns = df.columns.str.title()
df.columns


Index(['Title', 'Property_Size', 'Bedrooms', 'Bathrooms', 'Furnishing',
       'Region', 'Region_Name', 'Region_Parent_Name', 'Is_Boost', 'Price'],
      dtype='str')

In [6]:
#Clean price

df['Price'] = (
    df['Price']
    .astype(str)
    .replace(r"[₦,]", "", regex=True)
    .astype(float)
)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               1980 non-null   str    
 1   Property_Size       1980 non-null   int64  
 2   Bedrooms            1980 non-null   int64  
 3   Bathrooms           1980 non-null   int64  
 4   Furnishing          1980 non-null   str    
 5   Region              1980 non-null   str    
 6   Region_Name         1980 non-null   str    
 7   Region_Parent_Name  1978 non-null   str    
 8   Is_Boost            1980 non-null   str    
 9   Price               1980 non-null   float64
dtypes: float64(1), int64(3), str(6)
memory usage: 344.8 KB


In [7]:
#Dealing with Outliers

Q1 = df['Price'].quantile(0.25)
Q3 = df['Price'].quantile(0.75)

IQR = Q3 - Q1

#Finding the Lower and upper bounds
lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR

outliers = (df['Price'] < lower_bound) | (df['Price'] > upper_bound)

print('Outliers:', outliers.sum())

Outliers: 194


In [8]:
#Excluding Outliers from dataframe using ""

lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR

new_df = df[
    (df['Price'] > lower_bound) & (df['Price'] < upper_bound)
]

new_df.info()

<class 'pandas.DataFrame'>
Index: 1786 entries, 0 to 1979
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               1786 non-null   str    
 1   Property_Size       1786 non-null   int64  
 2   Bedrooms            1786 non-null   int64  
 3   Bathrooms           1786 non-null   int64  
 4   Furnishing          1786 non-null   str    
 5   Region              1786 non-null   str    
 6   Region_Name         1786 non-null   str    
 7   Region_Parent_Name  1784 non-null   str    
 8   Is_Boost            1786 non-null   str    
 9   Price               1786 non-null   float64
dtypes: float64(1), int64(3), str(6)
memory usage: 325.7 KB


In [9]:
#Convert furnishing type to consistent categories

new_df['Furnishing'] = new_df['Furnishing'].astype(str).str.title()

new_df.head()

,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,"3bdrm Penthouse in Brownstone Estate, Ikate fo...",150,3,3,Semi-Furnished,"Lekki, Ikate",Ikate,Lekki,False,250000000.00
1,"4bdrm Duplex in Redemption Estate, Owerri for ...",400,4,5,Unfurnished,"Imo State, Owerri",Owerri,Imo State,premium,200000000.00
2,6bdrm Bungalow in Ikorodu Garage for sale,5000,6,6,Unfurnished,"Ikorodu, Ikorodu Garage",Ikorodu Garage,Ikorodu,False,65000000.00
3,4bdrm Duplex in Opebi for sale,225,4,4,Semi-Furnished,"Ikeja, Opebi",Opebi,Ikeja,enterprise,450000000.00
4,"3bdrm House in Treasure Hilltop, Akesan for sale",300,3,4,Unfurnished,"Alimosho, Akesan",Akesan,Alimosho,False,85000000.00


In [26]:

#Convert is_boost type to consistent categories

new_df['Is_Boost'] = new_df['Is_Boost'].astype(str).str.title()

new_df.head()


,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,"3bdrm Penthouse in Brownstone Estate, Ikate fo...",150,3,3,Semi-Furnished,"Lekki, Ikate",Ikate,Lagos State,False,250000000.00
1,"4bdrm Duplex in Redemption Estate, Owerri for ...",400,4,5,Unfurnished,"Imo State, Owerri",Owerri,Imo State,Premium,200000000.00
2,6bdrm Bungalow in Ikorodu Garage for sale,5000,6,6,Unfurnished,"Ikorodu, Ikorodu Garage",Ikorodu Garage,Lagos State,False,65000000.00
3,4bdrm Duplex in Opebi for sale,225,4,4,Semi-Furnished,"Ikeja, Opebi",Opebi,Lagos State,Enterprise,450000000.00
4,"3bdrm House in Treasure Hilltop, Akesan for sale",300,3,4,Unfurnished,"Alimosho, Akesan",Akesan,Lagos State,False,85000000.00


In [27]:
new_df['Is_Boost'] = new_df['Is_Boost'].replace('Vip_Gold', 'VIP Gold')


In [28]:
new_df['Is_Boost'] = new_df['Is_Boost'].replace('Vip', 'VIP')
new_df.head()

,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,"3bdrm Penthouse in Brownstone Estate, Ikate fo...",150,3,3,Semi-Furnished,"Lekki, Ikate",Ikate,Lagos State,False,250000000.00
1,"4bdrm Duplex in Redemption Estate, Owerri for ...",400,4,5,Unfurnished,"Imo State, Owerri",Owerri,Imo State,Premium,200000000.00
2,6bdrm Bungalow in Ikorodu Garage for sale,5000,6,6,Unfurnished,"Ikorodu, Ikorodu Garage",Ikorodu Garage,Lagos State,False,65000000.00
3,4bdrm Duplex in Opebi for sale,225,4,4,Semi-Furnished,"Ikeja, Opebi",Opebi,Lagos State,Enterprise,450000000.00
4,"3bdrm House in Treasure Hilltop, Akesan for sale",300,3,4,Unfurnished,"Alimosho, Akesan",Akesan,Lagos State,False,85000000.00


In [29]:
total_duplicates = new_df.duplicated().sum()
print(total_duplicates)

0


In [30]:
new_df = new_df.drop_duplicates()
new_df.duplicated().sum()

np.int64(0)

In [31]:

new_df['Region_Parent_Name'] = new_df['Region_Parent_Name'].replace(['Lekki', 'Ikorodu', 'Ajah', 'Ojodu', 'Ikeja', 'Ipaja', 'Egbeda', 'Agege', 'Ibeju', 'Ogba', 
                                      'Magodo', 'Gbagada', 'Shomolu', 'Ojo', 'Lagos Island (Eko)', 'Ogudu', 'Isolo', 'Kosofe', 
                                      'Ikotun/Igando', 'Egbe/Idimu', 'Yaba', 'Ifako-Ijaiye', 'Alimosho', 'Ikoyi', 'Badagry', 
                                      'Apapa', 'Surulere', 'Oshodi', 'Ilupeju', 'Ojota', 'Sagamu', 'Epe', 'Mushin', 'Amuwo-Odofin', 'Ejigbo', 'Maryland', 'Victoria Island'], 'Lagos State')

new_df['Region_Parent_Name'].unique()

<ArrowStringArray>
[  'Lagos State',     'Imo State',   'Abuja (FCT)',    'Osun State',
     'Oyo State', 'Port-Harcourt',   'Ekiti State',    'Ogun State',
 'Bayelsa State',    'Ondo State',  'Rivers State',   'Enugu State',
   'Kwara State',     'Edo State',  'Kaduna State',  'Jigawa State',
   'Niger State',    'Abia State',   'Delta State']
Length: 19, dtype: str

In [32]:
new_df['Region_Parent_Name'] = new_df['Region_Parent_Name'].replace(['Apo District', 'Lugbe District', 'Bwari', 'Gwarinpa', 'Gwagwa', 'Katampe', 'Jiwa', 'Garki 1', 'Wuse'], 'Abuja (FCT)')

new_df['Region_Parent_Name'].unique()

<ArrowStringArray>
[  'Lagos State',     'Imo State',   'Abuja (FCT)',    'Osun State',
     'Oyo State', 'Port-Harcourt',   'Ekiti State',    'Ogun State',
 'Bayelsa State',    'Ondo State',  'Rivers State',   'Enugu State',
   'Kwara State',     'Edo State',  'Kaduna State',  'Jigawa State',
   'Niger State',    'Abia State',   'Delta State']
Length: 19, dtype: str

In [33]:
new_df['Region_Parent_Name'] = new_df['Region_Parent_Name'].replace('Ibadan', 'Oyo State')
new_df['Region_Parent_Name'].value_counts()

Region_Parent_Name
Lagos State      1047
Abuja (FCT)       430
Oyo State         128
Port-Harcourt      34
Edo State          22
Ogun State         17
Rivers State       16
Imo State          12
Enugu State         7
Ondo State          6
Osun State          3
Kwara State         2
Ekiti State         1
Bayelsa State       1
Kaduna State        1
Jigawa State        1
Niger State         1
Abia State          1
Delta State         1
Name: count, dtype: int64

In [34]:
new_df.info()

<class 'pandas.DataFrame'>
Index: 1731 entries, 0 to 1979
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               1731 non-null   str    
 1   Property_Size       1731 non-null   int64  
 2   Bedrooms            1731 non-null   int64  
 3   Bathrooms           1731 non-null   int64  
 4   Furnishing          1731 non-null   str    
 5   Region              1731 non-null   str    
 6   Region_Name         1731 non-null   str    
 7   Region_Parent_Name  1731 non-null   str    
 8   Is_Boost            1731 non-null   str    
 9   Price               1731 non-null   float64
dtypes: float64(1), int64(3), str(6)
memory usage: 321.1 KB


In [35]:
new_df = new_df.dropna()

new_df.info()

<class 'pandas.DataFrame'>
Index: 1731 entries, 0 to 1979
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               1731 non-null   str    
 1   Property_Size       1731 non-null   int64  
 2   Bedrooms            1731 non-null   int64  
 3   Bathrooms           1731 non-null   int64  
 4   Furnishing          1731 non-null   str    
 5   Region              1731 non-null   str    
 6   Region_Name         1731 non-null   str    
 7   Region_Parent_Name  1731 non-null   str    
 8   Is_Boost            1731 non-null   str    
 9   Price               1731 non-null   float64
dtypes: float64(1), int64(3), str(6)
memory usage: 321.1 KB


In [36]:

new_df.to_csv('data/jiji_housing_cleaned.csv', index=False)
new_df.info()



<class 'pandas.DataFrame'>
Index: 1731 entries, 0 to 1979
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               1731 non-null   str    
 1   Property_Size       1731 non-null   int64  
 2   Bedrooms            1731 non-null   int64  
 3   Bathrooms           1731 non-null   int64  
 4   Furnishing          1731 non-null   str    
 5   Region              1731 non-null   str    
 6   Region_Name         1731 non-null   str    
 7   Region_Parent_Name  1731 non-null   str    
 8   Is_Boost            1731 non-null   str    
 9   Price               1731 non-null   float64
dtypes: float64(1), int64(3), str(6)
memory usage: 321.1 KB


In [37]:
#Exploratory Data Analysis (EDA)


In [38]:

average_price = round(new_df['Price'].mean(), 2)
print('Housing Average Price(#)', average_price)


Housing Average Price(#) 291757336.8


In [39]:

round(new_df.groupby('Region_Parent_Name')['Price'].mean(), 2)

Region_Parent_Name
Abia State      210000000.00
Abuja (FCT)     300354651.16
Bayelsa State   150000000.00
Delta State     200000000.00
Edo State       111227272.73
Ekiti State      35000000.00
Enugu State     212857142.86
Imo State       204625000.00
Jigawa State    870000000.00
Kaduna State     50000000.00
Kwara State      22500000.00
Lagos State     322764708.69
Niger State      26000000.00
Ogun State       69499999.94
Ondo State       80000000.00
Osun State       60000000.00
Oyo State       135568750.00
Port-Harcourt   221058823.53
Rivers State    203500000.00
Name: Price, dtype: float64

In [40]:
property_counts = new_df['Property_Size'].value_counts().sort_values(ascending=False).head(20).reset_index(name='Count')
print(property_counts)

    Property_Size  Count
0             500    307
1             300    145
2             600    122
3            1000    118
4             400    104
5             250     91
6             100     86
7             450     79
8             350     64
9             200     53
10            700     53
11            150     40
12            800     36
13            650     23
14            750     20
15            900     18
16            550     13
17           1500     13
18            320     11
19            648     10


In [41]:
region_counts = new_df['Region_Parent_Name'].value_counts().sort_values(ascending=False).head(20).reset_index(name='Count')
print(region_counts)

   Region_Parent_Name  Count
0         Lagos State   1047
1         Abuja (FCT)    430
2           Oyo State    128
3       Port-Harcourt     34
4           Edo State     22
5          Ogun State     17
6        Rivers State     16
7           Imo State     12
8         Enugu State      7
9          Ondo State      6
10         Osun State      3
11        Kwara State      2
12        Ekiti State      1
13      Bayelsa State      1
14       Kaduna State      1
15       Jigawa State      1
16        Niger State      1
17         Abia State      1
18        Delta State      1


In [42]:
new_df.groupby('Region_Parent_Name')['Price'].mean().round(2).sort_values(ascending=False).reset_index(name='Price')

,Region_Parent_Name,Price
0,Jigawa State,870000000.00
1,Lagos State,322764708.69
2,Abuja (FCT),300354651.16
3,Port-Harcourt,221058823.53
4,Enugu State,212857142.86
5,Abia State,210000000.00
6,Imo State,204625000.00
7,Rivers State,203500000.00
8,Delta State,200000000.00
9,Bayelsa State,150000000.00


In [48]:
#Which regions dominate premium property sales?
vip_df = new_df[new_df['Is_Boost'] == 'Premium']

region_totals = vip_df['Region_Parent_Name'].value_counts()

print(region_totals)

Region_Parent_Name
Lagos State    6
Oyo State      5
Imo State      3
Ogun State     1
Name: count, dtype: int64


In [49]:
#Are furnished apartments more expensive on average?

furnished = new_df.groupby('Furnishing')['Price'].mean().sort_values(ascending=False).head(20).reset_index(name='Average')
print(furnished)



       Furnishing      Average
0       Furnished 319276785.71
1  Semi-Furnished 307402881.35
2     Unfurnished 274968647.76


In [50]:
#Do boosted/enterprise listings have higher prices?

boosted = new_df.groupby('Is_Boost')['Price'].sum().sort_values(ascending=False).head(20).reset_index(name='Total Price')
print(boosted)

     Is_Boost     Total Price
0  Enterprise 276785550000.00
1     Diamond 203007499999.00
2         VIP   8559999999.00
3       False   7660900000.00
4    Vip Gold   4773000000.00
5       Basic   2128000000.00
6     Premium   2117000000.00
